# REBA Video Pipeline - CSV + Annotated Video + Grafik

## TÜBİTAK Projesi - Sanayi İçin Derin Öğrenme Tabanlı Ergonomik Karar Destek Sistemi

**Bu notebook hem video dosyası hem webcam ile çalışır.**

### Çıktılar:
1. **Annotated Video** - İskelet + REBA skor overlay'li MP4
2. **CSV** - Kare kare REBA skoru, step skorları, tahmin edilen açılar
3. **Grafikler** - REBA zaman serisi, risk dağılımı, step box plot, açı grafikleri
4. **JSON Özet** - İstatistikler ve risk dağılımı

### Pipeline:
```
Video/Webcam → YOLO Pose → Açı Hesaplama → FT-Transformer → REBA Skor → Çıktılar
```

## 1. Imports ve Konfigürasyon

In [ ]:
import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from Utils.Functions import add_pose_angles
from Utils.Models import (
    FeatureTokenizer, FeaturePositionalEmbedding,
    AttentionPooling, ResidualMLPBlock,
    build_advanced_ft_transformer_regression,
    predict_advanced_multioutput
)
from Utils.Reba import (
    rules, compute_reba_scores,
    get_reba_tableA_score, get_reba_tableB_score, get_reba_tableC_score,
    get_force_load_score, get_coupling_score, get_activity_score,
    classify_reba_risk, clamp
)
from Utils.RebaCalibration import compute_reba_deviations

print(f'TensorFlow: {tf.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ============================================================
# KONFIGÜRASYON - Buradan ayarlayın
# ============================================================

# Kaynak: Video dosya yolu VEYA webcam indeksi (0, 1, ...)
SOURCE = 0  # Webcam için 0, video için dosya yolu yazın
# SOURCE = r"C:\path\to\video.mp4"  # Video dosyası örneği

# Çıktı klasörü
OUTPUT_DIR = "Output_Video"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model yolları
YOLO_MODEL_PATH = "yolo11n-pose.pt"
MODEL_PATH = r"BestModel\best_ft_transformer_multioutput_compatible.h5"
SCALER_PATH = r"BestModel\ft_transformer_scaler.pkl"
META_PATH = r"BestModel\advanced_ft_transformer_meta.json"

# Parametreler
YOLO_CONF = 0.25           # YOLO güven eşiği
SKIP_FRAMES = 0            # Kare atlama (0=her kare, 2=her 3. kare)
MAX_FRAMES = None          # Maks kare (None=tümü, webcam için None önerilir)
SAVE_VIDEO = True          # Annotated video kaydet
DISPLAY_LIVE = False       # Notebook'ta canlı gösterim (yavaşlatır)

print(f'Kaynak: {SOURCE}')
print(f'Çıktı: {OUTPUT_DIR}')

## 2. Sabitler ve Yardımcı Tanımlar

In [ ]:
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

BASE_INPUT_COLS = []
for name in KEYPOINT_NAMES:
    BASE_INPUT_COLS.extend([f"{name}_x", f"{name}_y", f"{name}_z"])

TARGET_COLS = [
    'elbowL_yz_deg', 'elbowR_yz_deg',
    'head_vs_neck_xz_deg', 'head_vs_neck_yz_deg',
    'kneeL_xz_deg', 'kneeL_yz_deg', 'kneeR_xz_deg', 'kneeR_yz_deg',
    'shoulderL_vs_upperarmL_xz_deg', 'shoulderL_vs_upperarmL_yz_deg',
    'shoulderR_vs_upperarmR_xz_deg', 'shoulderR_vs_upperarmR_yz_deg',
    'spine1_vs_spine2_yz_deg', 'spine2_vs_spine3_yz_deg', 'spine3_vs_spine4_yz_deg',
    'thighL_vs_spine1_yz_deg', 'thighR_vs_spine1_yz_deg',
    'wristL_yz_deg', 'wristR_yz_deg',
    'head_q_x', 'head_q_y', 'head_q_z',
    'upperarmL_q_x', 'upperarmL_q_y', 'upperarmL_q_z',
    'upperarmR_q_x', 'upperarmR_q_y', 'upperarmR_q_z',
    'handL_q_x', 'handL_q_y', 'handL_q_z',
    'handR_q_x', 'handR_q_y', 'handR_q_z',
    'thighL_q_x', 'thighL_q_y', 'thighL_q_z',
    'thighR_q_x', 'thighR_q_y', 'thighR_q_z',
    'shinL_q_x', 'shinL_q_y', 'shinL_q_z',
    'shinR_q_x', 'shinR_q_y', 'shinR_q_z',
    'spine1_q_x', 'spine1_q_y', 'spine1_q_z',
    'spine2_q_x', 'spine2_q_y', 'spine2_q_z',
    'spine3_q_x', 'spine3_q_y', 'spine3_q_z',
    'spine4_q_x', 'spine4_q_y', 'spine4_q_z'
]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]

RISK_COLORS = {
    "Negligible risk": (0, 200, 0),
    "Low risk": (0, 255, 100),
    "Medium risk": (0, 200, 255),
    "High risk": (0, 100, 255),
    "Very high risk": (0, 0, 255),
}

print(f'Input kolonları: {len(BASE_INPUT_COLS)}')
print(f'Target kolonları: {len(TARGET_COLS)}')

## 3. Model Yükleme

In [ ]:
print('Modeller yükleniyor...')

# YOLO
yolo_model = YOLO(YOLO_MODEL_PATH)
print(f'  YOLO yüklendi: {YOLO_MODEL_PATH}')

# Scaler
scaler = joblib.load(SCALER_PATH)
print(f'  Scaler yüklendi: {SCALER_PATH}')

# FT-Transformer
with open(META_PATH, 'r') as f:
    meta_info = json.load(f)

ft_model = build_advanced_ft_transformer_regression(
    n_features=meta_info['input_dim'],
    n_targets=meta_info['n_targets'],
    d_token=96, n_blocks=6, n_heads=8,
    dropout=0.20, ff_mult=4,
    shared_mlp_units=(256, 128),
    per_target_hidden=64,
    use_uncertainty=meta_info['use_uncertainty']
)

_dummy = np.zeros((1, meta_info['input_dim']), dtype='float32')
_ = ft_model(_dummy)
ft_model.load_weights(MODEL_PATH)
print(f'  FT-Transformer yüklendi: {MODEL_PATH}')
print(f'  Input: {meta_info["input_dim"]}, Output: {meta_info["n_targets"]}')
print('\nTüm modeller hazır!')

## 4. Analiz Fonksiyonları

In [ ]:
def process_frame(frame):
    """Tek kareyi analiz eder. REBA sonucu ve keypoint'leri döndürür."""
    results = yolo_model.predict(source=frame, conf=YOLO_CONF, save=False, verbose=False)
    
    if not results or results[0].keypoints is None:
        return None, None
    
    kp_data = results[0].keypoints.data.cpu().numpy()
    if kp_data.shape[0] == 0:
        return None, None
    
    pts = kp_data[0]
    keypoints = [[float(kp[0]), float(kp[1]), float(kp[2])] for kp in pts]
    
    # DataFrame
    input_row = {}
    for i, name in enumerate(KEYPOINT_NAMES):
        x, y, z = keypoints[i] if i < len(keypoints) else (0.0, 0.0, 0.0)
        input_row[f"{name}_x"] = x
        input_row[f"{name}_y"] = y
        input_row[f"{name}_z"] = z
    
    df_kp = pd.DataFrame([input_row])
    df_kp.columns = BASE_INPUT_COLS
    df_input = add_pose_angles(df_kp, add_planar_angles=True)
    
    # FT-Transformer
    X_input = df_input.values.astype('float32')
    X_scaled = scaler.transform(X_input)
    y_pred = predict_advanced_multioutput(ft_model, X_scaled, use_uncertainty=False, batch_size=1)
    df_pred = pd.DataFrame(y_pred, columns=TARGET_COLS)
    
    # REBA - kalibre edilmis deviation'lar
    df_reba = compute_reba_deviations(df_pred)
    
    df_sc = compute_reba_scores(df_reba, rules)
    df_sc = df_sc[["Score_Step1", "Score_Step2", "Score_Step3",
                   "Score_Step4", "Score_Step5", "Score_Step6"]].copy()
    
    s1 = int(df_sc["Score_Step1"].iloc[0])
    s2 = int(df_sc["Score_Step2"].iloc[0])
    s3 = int(df_sc["Score_Step3"].iloc[0])
    s4 = int(df_sc["Score_Step4"].iloc[0])
    s5 = int(df_sc["Score_Step5"].iloc[0])
    s6 = int(df_sc["Score_Step6"].iloc[0])
    
    tA = get_reba_tableA_score(neck=clamp(s1,1,3), trunk=clamp(s2,1,5), legs=clamp(s3,1,4))
    scoreA = tA + get_force_load_score(0, False)
    tB = get_reba_tableB_score(upper_arm=clamp(s4,1,6), lower_arm=clamp(s5,1,2), wrist=clamp(s6,1,3))
    scoreB = tB + get_coupling_score(0)
    tC = get_reba_tableC_score(clamp(scoreA,1,12), clamp(scoreB,1,12))
    final_reba = tC + get_activity_score(False, False, False)
    risk_level = classify_reba_risk(final_reba)
    
    result_dict = {
        "step1_neck": s1, "step2_trunk": s2, "step3_legs": s3,
        "step4_upper_arm": s4, "step5_elbow": s5, "step6_wrist": s6,
        "score_a": int(scoreA), "score_b": int(scoreB),
        "table_c": int(tC), "final_reba": int(final_reba),
        "risk_level": risk_level,
        "predicted_angles": {col: float(df_pred[col].iloc[0]) for col in TARGET_COLS[:19]},
    }
    
    return result_dict, keypoints


def draw_skeleton(frame, keypoints, conf_threshold=0.3):
    """Iskelet cizer."""
    for (i, j) in SKELETON_CONNECTIONS:
        if i >= len(keypoints) or j >= len(keypoints):
            continue
        x1, y1, c1 = keypoints[i]
        x2, y2, c2 = keypoints[j]
        if c1 < conf_threshold or c2 < conf_threshold:
            continue
        cv2.line(frame, (int(x1),int(y1)), (int(x2),int(y2)), (0,255,0), 2, cv2.LINE_AA)
    
    for x, y, c in keypoints:
        if c < conf_threshold:
            continue
        cv2.circle(frame, (int(x),int(y)), 4, (0,0,255), -1, cv2.LINE_AA)
    return frame


def draw_reba_overlay(frame, result_dict):
    """REBA skor overlay'i cizer."""
    if result_dict is None:
        cv2.putText(frame, "Kisi tespit edilemedi", (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
        return frame
    
    h, w = frame.shape[:2]
    final_reba = result_dict["final_reba"]
    risk_level = result_dict["risk_level"]
    risk_color = RISK_COLORS.get(risk_level, (255,255,255))
    
    # Arka plan kutusu
    overlay = frame.copy()
    cv2.rectangle(overlay, (10,10), (330,200), (0,0,0), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
    
    y = 35
    cv2.putText(frame, "REBA ANALIZI", (20,y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    y += 40
    cv2.putText(frame, f"SKOR: {final_reba}", (20,y), cv2.FONT_HERSHEY_SIMPLEX, 1.2, risk_color, 3)
    y += 30
    cv2.putText(frame, f"Risk: {risk_level}", (20,y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, risk_color, 2)
    y += 30
    cv2.putText(frame, f"Boyun:{result_dict['step1_neck']} Govde:{result_dict['step2_trunk']} Bacak:{result_dict['step3_legs']}",
                (20,y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    y += 22
    cv2.putText(frame, f"UstKol:{result_dict['step4_upper_arm']} Dirsek:{result_dict['step5_elbow']} Bilek:{result_dict['step6_wrist']}",
                (20,y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    y += 22
    cv2.putText(frame, f"Score A:{result_dict['score_a']}  Score B:{result_dict['score_b']}",
                (20,y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    
    return frame


print('Analiz fonksiyonlari hazir.')

## 5. Video Analizi - Ana Pipeline

In [ ]:
# Video kaynağını aç
is_webcam = isinstance(SOURCE, int)
source_name = f"webcam_{SOURCE}" if is_webcam else os.path.splitext(os.path.basename(SOURCE))[0]

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError(f"Video açılamadı: {SOURCE}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if not is_webcam else -1

print(f'Kaynak: {"Webcam" if is_webcam else SOURCE}')
print(f'Çözünürlük: {frame_w}x{frame_h} @ {fps:.1f} FPS')
if total_frames > 0:
    print(f'Toplam kare: {total_frames} ({total_frames/fps:.1f} sn)')

# Video writer
video_writer = None
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
if SAVE_VIDEO:
    video_out_path = os.path.join(OUTPUT_DIR, f'{source_name}_reba_{timestamp}.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(video_out_path, fourcc, fps, (frame_w, frame_h))
    print(f'Video çıktı: {video_out_path}')

In [ ]:
# ANA ANALİZ DÖNGÜSÜ
csv_rows = []
frame_idx = 0
processed_count = 0
start_time = time.time()
last_result = None

print('Analiz başlıyor...')
print('Webcam kullanıyorsanız durdurmak için Kernel > Interrupt kullanın.')
print('-' * 60)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            if is_webcam:
                continue
            break
        
        frame_idx += 1
        if MAX_FRAMES and frame_idx > MAX_FRAMES:
            break
        
        should_analyze = (SKIP_FRAMES == 0) or (frame_idx % (SKIP_FRAMES + 1) == 0)
        
        if should_analyze:
            result_dict, keypoints = process_frame(frame)
            last_result = result_dict
            processed_count += 1
            
            # CSV satırı
            row = {"frame": frame_idx, "timestamp_sec": frame_idx / fps}
            if result_dict:
                row.update({
                    "final_reba": result_dict["final_reba"],
                    "risk_level": result_dict["risk_level"],
                    "step1_neck": result_dict["step1_neck"],
                    "step2_trunk": result_dict["step2_trunk"],
                    "step3_legs": result_dict["step3_legs"],
                    "step4_upper_arm": result_dict["step4_upper_arm"],
                    "step5_elbow": result_dict["step5_elbow"],
                    "step6_wrist": result_dict["step6_wrist"],
                    "score_a": result_dict["score_a"],
                    "score_b": result_dict["score_b"],
                })
                row.update(result_dict.get("predicted_angles", {}))
            else:
                row["final_reba"] = None
                row["risk_level"] = "Tespit edilemedi"
            
            csv_rows.append(row)
            
            # İlerleme
            if processed_count % 50 == 0:
                elapsed = time.time() - start_time
                proc_fps = processed_count / elapsed if elapsed > 0 else 0
                reba_val = result_dict['final_reba'] if result_dict else '-'
                if total_frames > 0:
                    pct = frame_idx / total_frames * 100
                    print(f'  Kare {frame_idx}/{total_frames} ({pct:.0f}%) | REBA: {reba_val} | {proc_fps:.1f} FPS')
                else:
                    print(f'  Kare {frame_idx} | REBA: {reba_val} | {proc_fps:.1f} FPS')
        else:
            result_dict = last_result
            keypoints = None
        
        # Annotated frame
        annotated = frame.copy()
        if keypoints:
            annotated = draw_skeleton(annotated, keypoints)
        annotated = draw_reba_overlay(annotated, result_dict)
        
        if video_writer:
            video_writer.write(annotated)

except KeyboardInterrupt:
    print('\nDurduruldu.')

finally:
    cap.release()
    if video_writer:
        video_writer.release()

total_time = time.time() - start_time
print(f'\n{"="*60}')
print(f'  ANALİZ TAMAMLANDI')
print(f'  Süre: {total_time:.1f} sn | Kare: {processed_count} | FPS: {processed_count/total_time:.1f}')
print(f'{"="*60}')

## 6. Sonuçları Kaydetme

In [ ]:
# CSV Kaydet
df_results = pd.DataFrame(csv_rows)
csv_path = os.path.join(OUTPUT_DIR, f'{source_name}_reba_results_{timestamp}.csv')
df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'CSV kaydedildi: {csv_path}')
print(f'Satır sayısı: {len(df_results)}')

# İlk birkaç satır
display(df_results[['frame', 'timestamp_sec', 'final_reba', 'risk_level',
                    'step1_neck', 'step2_trunk', 'step3_legs',
                    'step4_upper_arm', 'step5_elbow', 'step6_wrist']].head(10))

## 7. Grafikler

In [ ]:
df_valid = df_results[df_results['final_reba'].notna()].copy()

if df_valid.empty:
    print('Geçerli veri yok!')
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'REBA Ergonomik Risk Analizi - {source_name}', fontsize=14, fontweight='bold')
    
    # 1. REBA Zaman Serisi
    ax = axes[0, 0]
    t = df_valid['timestamp_sec'].values
    reba = df_valid['final_reba'].values
    ax.plot(t, reba, 'b-', linewidth=1, alpha=0.7)
    ax.axhspan(0, 1, alpha=0.1, color='green')
    ax.axhspan(1, 3, alpha=0.1, color='limegreen')
    ax.axhspan(3, 7, alpha=0.1, color='orange')
    ax.axhspan(7, 10, alpha=0.1, color='orangered')
    ax.axhspan(10, 15, alpha=0.1, color='red')
    if len(reba) > 10:
        w = min(30, len(reba)//3)
        ax.plot(t, pd.Series(reba).rolling(w, min_periods=1).mean(), 'r-', lw=2, label=f'Ort({w})')
    ax.set_xlabel('Zaman (sn)'); ax.set_ylabel('REBA')
    ax.set_title('REBA Skoru - Zaman Serisi')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 2. Risk Dağılımı
    ax = axes[0, 1]
    risk_counts = df_valid['risk_level'].value_counts()
    colors_pie = {'Negligible risk':'#2ecc71','Low risk':'#a3d977',
                  'Medium risk':'#f39c12','High risk':'#e67e22','Very high risk':'#e74c3c'}
    labels = []; sizes = []; cols = []
    for r in ['Negligible risk','Low risk','Medium risk','High risk','Very high risk']:
        if r in risk_counts.index:
            labels.append(r.replace(' risk',''))
            sizes.append(risk_counts[r])
            cols.append(colors_pie[r])
    if sizes:
        ax.pie(sizes, labels=labels, colors=cols, autopct='%1.1f%%', startangle=90)
    ax.set_title('Risk Dağılımı')
    
    # 3. Step Box Plot
    ax = axes[1, 0]
    step_cols = ['step1_neck','step2_trunk','step3_legs','step4_upper_arm','step5_elbow','step6_wrist']
    step_labels = ['Boyun','Gövde','Bacak','Üst Kol','Dirsek','Bilek']
    data = [df_valid[c].dropna().values for c in step_cols if c in df_valid.columns]
    if data:
        bp = ax.boxplot(data, labels=step_labels[:len(data)], patch_artist=True)
        bpc = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
        for patch, color in zip(bp['boxes'], bpc):
            patch.set_facecolor(color); patch.set_alpha(0.6)
    ax.set_ylabel('Skor'); ax.set_title('Step Skorları'); ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Score A & B
    ax = axes[1, 1]
    if 'score_a' in df_valid.columns:
        ax.plot(t, df_valid['score_a'].values, 'b-', lw=1.5, label='Score A', alpha=0.8)
        ax.plot(t, df_valid['score_b'].values, 'r-', lw=1.5, label='Score B', alpha=0.8)
    ax.set_xlabel('Zaman (sn)'); ax.set_ylabel('Skor')
    ax.set_title('Score A & B'); ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, f'{source_name}_charts_{timestamp}.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grafik kaydedildi: {chart_path}')

In [ ]:
# Açı değişim grafiği
if not df_valid.empty and len(df_valid) > 2:
    angle_cols = [c for c in df_valid.columns if c.endswith('_deg') and c in TARGET_COLS[:19]]
    
    if angle_cols:
        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        fig.suptitle('Tahmin Edilen Vücut Açıları', fontsize=12, fontweight='bold')
        t = df_valid['timestamp_sec'].values
        
        # Üst gövde
        upper = [c for c in angle_cols if 'spine' in c or 'neck' in c or 'head' in c]
        for col in upper:
            if col in df_valid.columns:
                axes[0].plot(t, df_valid[col].values, lw=1, label=col.replace('_deg',''), alpha=0.8)
        axes[0].set_ylabel('Derece'); axes[0].set_title('Üst Gövde / Boyun')
        axes[0].legend(loc='upper right', fontsize=7); axes[0].grid(True, alpha=0.3)
        
        # Kol
        arm = [c for c in angle_cols if 'elbow' in c or 'wrist' in c or 'shoulder' in c]
        for col in arm:
            if col in df_valid.columns:
                axes[1].plot(t, df_valid[col].values, lw=1, label=col.replace('_deg',''), alpha=0.8)
        axes[1].set_ylabel('Derece'); axes[1].set_title('Kol / Dirsek / Bilek')
        axes[1].legend(loc='upper right', fontsize=7); axes[1].grid(True, alpha=0.3)
        
        # Bacak
        leg = [c for c in angle_cols if 'knee' in c or 'thigh' in c]
        for col in leg:
            if col in df_valid.columns:
                axes[2].plot(t, df_valid[col].values, lw=1, label=col.replace('_deg',''), alpha=0.8)
        axes[2].set_xlabel('Zaman (sn)'); axes[2].set_ylabel('Derece')
        axes[2].set_title('Bacak / Diz'); axes[2].legend(loc='upper right', fontsize=7)
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        angle_path = os.path.join(OUTPUT_DIR, f'{source_name}_angles_{timestamp}.png')
        plt.savefig(angle_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Açı grafiği kaydedildi: {angle_path}')

## 8. Özet Rapor

In [ ]:
if not df_valid.empty:
    reba_scores = df_valid['final_reba'].values
    
    summary = {
        'source': source_name,
        'analysis_date': datetime.now().isoformat(),
        'total_frames': len(df_valid),
        'duration_sec': float(total_time),
        'fps': float(fps),
        'reba_stats': {
            'mean': float(np.mean(reba_scores)),
            'median': float(np.median(reba_scores)),
            'std': float(np.std(reba_scores)),
            'min': int(np.min(reba_scores)),
            'max': int(np.max(reba_scores)),
        },
        'risk_distribution': df_valid['risk_level'].value_counts().to_dict(),
        'high_risk_pct': f"{(reba_scores >= 8).sum() / len(reba_scores) * 100:.1f}%",
    }
    
    summary_path = os.path.join(OUTPUT_DIR, f'{source_name}_summary_{timestamp}.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    
    print('=' * 60)
    print('  ANALIZ OZETI')
    print('=' * 60)
    print(f"  Ortalama REBA: {summary['reba_stats']['mean']:.1f}")
    print(f"  Medyan REBA:   {summary['reba_stats']['median']:.1f}")
    print(f"  Min / Max:     {summary['reba_stats']['min']} / {summary['reba_stats']['max']}")
    print(f"  Yüksek Risk:   {summary['high_risk_pct']}")
    print(f'\n  Risk Dağılımı:')
    for level, count in sorted(summary['risk_distribution'].items()):
        pct = count / len(df_valid) * 100
        print(f'    {level}: {count} kare ({pct:.1f}%)')
    print('=' * 60)
    print(f'\nJSON: {summary_path}')
    if SAVE_VIDEO:
        print(f'Video: {video_out_path}')
    print(f'CSV:   {csv_path}')